# TS7: "Filtrado digital lineal de ECG"

## Alumnos: Irina Polack, Casandra Sotelino, Maylen Monterde y Lucila Suarez

En esta tarea semanal se nos pidio usar el archivo "ecg.mat" que contiene un registro electrocardiografico (ECG) como la señal para poder diseñar y aplicar los filtros digitales necesarios para poder mitigar las fuentes de contaminacion; como lo pueden ser, el ruido causado por el movimiento de los electrodos (Alta frecuencia), el ruido muscular (alta frecuenia) y el movimiento de la linea de base del ECG, inducido en parte por la respiracion (baja frecuencia). Tuvimos que establecer una plantilla de diseño para los filtros digitales, que fuimos modificando para no distorsionar la señal y poder filtrar las interferencias fuera del ancho de banda digital ("BW").


Los filtros digitales se dividen en dos grandes grupos los IIR y los FIR. 
Los IIR (respuesta infinita) se caracterizan por ser mas faciles de manipular al cumplir mas facil con la plantilla. Sin embargo, no siempre son estables por ser recursivos; usualmente no se usan en tiempo real. En este trabajo lo demostraremos utilazando Cheby tipo 2 y Butter. 
Este tipo de filtros tienen una contraparte analogica. Butter tiene una respuesta en magnitud analogica de la siguiente manera: $$\vert{}H_a(j\Omega)\vert{}^2 = \frac{1}{1 + \left(\frac{\Omega}{\Omega_c}\right)^{2N}}$$



Polos en el plano $s$:
Se continua analíticamente reemplazando $s = j\Omega$, lo que da $H_a(s)H_a(-s) = \frac{1}{1 + (-s^2/\Omega_c^2)^N}$. Los polos están dados por:$$s_k = \Omega_c \exp\left(j \left[ \frac{\pi}{2} + \frac{(2k - 1)\pi}{2N} \right]\right), \quad k = 1, 2, \dots, N \quad (\text{semiplano izquierdo})$$Transformación Bilineal (Mapeo a $z$):
Se aplica la sustitución con pre-warping $\Omega = \frac{2}{T} \tan\left(\frac{\omega}{2}\right)$:$$s = \frac{2}{T} \frac{1 - z^{-1}}{1 + z^{-1}}$$$$H(z) = H_a(s) \Big\vert{}_{s = \frac{2}{T} \frac{1 - z^{-1}}{1 + z^{-1}}}$$



Y el Chebby tipo 2 tiene una magnitud: 
$$\vert{}H_a(j\Omega)\vert{}^2 = \frac{1}{1 + \frac{1}{\epsilon^2 T_N^2(\Omega_r / \Omega)}} = \frac{\epsilon^2 T_N^2(\Omega_r / \Omega)}{1 + \epsilon^2 T_N^2(\Omega_r / \Omega)}$$donde $T_N(x)$ es el polinomio de Chebyshev de orden $N$, $\Omega_r = \Omega_s/\Omega_p$, y $\epsilon$ define la atenuación mínima en la banda de detención ($A_s = 10\log_{10}(1 + 1/\epsilon^2)$).Polos y Ceros en el plano $s$:Ceros (nulos de atenuación): Ocurren en las raíces de $T_N(\Omega_r / \Omega) = 0$:$$z_k = j \frac{\Omega_r}{\cos\left(\frac{(2k-1)\pi}{2N}\right)}, \quad k = 1, 2, \dots, N$$Polos: Inversos a los polos de Chebyshev Tipo I ($p_k^{TipoII} = \frac{\Omega_r \Omega_c}{p_k^{TipoI}}$).Mapeo Digital:
Se mapea al plano $z$ mediante la Transformación Bilineal con pre-enameramiento de frecuencia:$$H(z) = H_a(s) \Big\vert{}_{s = \frac{2}{T} \frac{1 - z^{-1}}{1 + z^{-1}}}$$



Por otro lado, tenemos el segundo tipo de filtro a analizar, que son los FIR (Finite Impulse Response). Estos siempre son estables, debido a que, sus polos estan en el origen; y usan un mayor orden. Se pueden usar en tiempo real y es mas simple que tengan una fase lineal, no distorisionan las señales. 
En el trabajo analizamos 3 tipos de FIR el de ventanas, el de cuadrados minimos y el de Remez.



El algoritmo de cuadrados mínimos es una técnica estadística que se utiliza para encontrar la mejor aproximación de un conjunto de datos minimizando la suma de los cuadrados de las diferencias entre los valores observados y los valores predichos por el modelo. En el contexto de filtros FIR, el objetivo es diseñar un filtro que minimice el error entre la respuesta del filtro y una respuesta deseada. 
Este tipo de filtros ofrecen una buena aproximación de la respuesta deseada, pero tienden a presentar un comportamiento diferente en comparación con otros métodos.

El objetivo es minimizar el error cuadrático entre la respuesta en frecuencia del filtro $H_R(\Omega)$ y la respuesta deseada $D(\Omega)$:

$$|E(\Omega)|^2 = \sum_{\Omega \in F} |D(\Omega) - H_R(\Omega)|^2$$

Como se verá más adelante, este problema se puede resolver de forma práctica mediante álgebra matricial en cualquier lenguaje de programación.

Se define la respuesta deseada $D(\Omega)$ en la banda de frecuencia, donde $\Omega$ es la frecuencia angular normalizada (de $0$ a $\pi$). La respuesta deseada del filtro en frecuencia $D(\Omega)$ puede definirse como:

$$D(\Omega) = \begin{cases} 
1 & \text{si } 0 \le \Omega \le \Omega_c \\ 
0 & \text{si } \Omega_c < \Omega \le 1.0 
\end{cases}$$






El algoritmo de Parks-McClellan (PM) es una herramienta poderosa utilizada para diseñar filtros FIR óptimos en términos de la minimización del error máximo en las bandas de frecuencia de interés. Se entiende al error como la diferencia entre una respuesta deseada (e.g. plantilla del filtro) y la obtenida por el algoritmo PM. Es decir que éste algoritmo distribuye el error de manera acotada en toda la banda de frecuencia, razón por la cual se lo suele denominar como error de Chebyshev o equirriple. La fundamentación matemática de este algoritmo se basa en el método de la alternancia de Remez, y una búsqueda iterativa de las frecuencias extremas que acotan la señal de error. 

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as sig
from matplotlib.patches import Rectangle 
import scipy.io as sio

ruta_npy = '.\ecg_sin_ruido.npy'
ecg_sin_ruido = np.load(ruta_npy)

print("Tipo:", type(ecg_sin_ruido))
print("Forma (shape):", ecg_sin_ruido.shape)

mat_path = './ECG_TP4.mat'
sio.whosmat(mat_path)
mat_struct = sio.loadmat(mat_path)
ecg_one_lead = mat_struct['ecg_lead']

FileNotFoundError: [Errno 2] No such file or directory: '.\\ecg_sin_ruido.npy'